In [ ]:
import optuna

study_name = ""  # replace with your actual study name
dataset = "Simulations_indep_traincontrol"
# dataset = "NCT00113763"
n_trials = 150
optuna_version_name = "ExMetrics2_bis_seedData{}_seedHPO{}".format(0, 10)
# optuna_version_name = "ExMetrics2_seedHPO{}".format(10)
n_samples = 600
n_features_bytype = 6
treatment_effect = 0.
name_config = "simu_N{}_nfeat{}_t{}".format(n_samples, n_features_bytype, int(treatment_effect))
generator_name = "HI-VAE_piecewise" # "HI-VAE_weibull" # "HI-VAE_piecewise" 
study_name_cluster = "/home/pchassat/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_{}_ntrials{}_{}_{}".format(dataset, name_config, n_trials, optuna_version_name, generator_name)
db_file = "/Users/pchassat/Documents/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_{}_ntrials{}_{}_{}.db".format(dataset, name_config, n_trials, optuna_version_name, generator_name)
# study_name_cluster = "/home/pchassat/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_traincontrol_{}_ntrials{}_{}_{}".format(dataset, dataset, n_trials, optuna_version_name, generator_name)
# db_file = "/Users/pchassat/Documents/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_traincontrol_{}_ntrials{}_{}_{}.db".format(dataset, dataset, n_trials, optuna_version_name, generator_name)
storage = f"sqlite:///{db_file}"
study = optuna.load_study(study_name=study_name_cluster, storage=storage)
names_objs = ["Survival curves dist","Identifiability score"]

In [47]:
from optuna.visualization import plot_parallel_coordinate, plot_slice, plot_param_importances

for i in range(len(names_objs)):
    plot_parallel_coordinate(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() # relationships between objectives and parameters

In [48]:
for i in range(len(names_objs)):
    plot_slice(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() # individual parameter effects

In [49]:
for i in range(len(names_objs)):
    print(f"Parameter importances for {names_objs[i]}:")
    plot_param_importances(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() 

Parameter importances for Survival curves dist:


Parameter importances for Identifiability score:


In [50]:
# Get all trials on the Pareto front
pareto_trials = study.best_trials  # these are Pareto optimal

for t in pareto_trials:
    print("Values:", t.values)
    print("Params:", t.params)
    print("------")

Values: [0.013359530752041847, 0.14666666666666667]
Params: {'lr': 0.001, 'batch_size': 32, 'z_dim': 110, 'y_dim': 160, 's_dim': 70}
------
Values: [0.020067247068494207, 0.10666666666666667]
Params: {'lr': 0.001, 'batch_size': 30, 'z_dim': 160, 'y_dim': 100, 's_dim': 190}
------
Values: [0.01531814758279918, 0.13]
Params: {'lr': 0.005, 'batch_size': 30, 'z_dim': 20, 'y_dim': 30, 's_dim': 200}
------
Values: [0.016522408543136188, 0.12]
Params: {'lr': 0.0001, 'batch_size': 120, 'z_dim': 110, 'y_dim': 130, 's_dim': 110}
------
Values: [0.020067247068494207, 0.10666666666666667]
Params: {'lr': 0.001, 'batch_size': 30, 'z_dim': 160, 'y_dim': 100, 's_dim': 190}
------
Values: [0.25444074757829926, 0.1]
Params: {'lr': 0.003, 'batch_size': 30, 'z_dim': 200, 'y_dim': 80, 's_dim': 40}
------
Values: [0.013359530752041847, 0.14666666666666667]
Params: {'lr': 0.001, 'batch_size': 32, 'z_dim': 110, 'y_dim': 160, 's_dim': 70}
------
Values: [0.25444074757829926, 0.1]
Params: {'lr': 0.003, 'batch_s

In [51]:
import pandas as pd

# df_pareto = pd.DataFrame([
#     {**t.params, **{names_objs[i]: v for i, v in enumerate(t.values)}}
#     for t in study.best_trials
# ])
# print(df_pareto)

df_pareto = pd.DataFrame([
    {
        "trial_number": t.number,          
        **t.params,
        **{names_objs[i]: v for i, v in enumerate(t.values)}
    }
    for t in study.best_trials
])

df_pareto.head(10)

,trial_number,lr,batch_size,z_dim,y_dim,s_dim,Survival curves dist,Identifiability score
0,35,0.0010,32,110,160,70,0.013360,0.146667
1,47,0.0010,30,160,100,190,0.020067,0.106667
2,51,0.0050,30,20,30,200,0.015318,0.130000
3,88,0.0001,120,110,130,110,0.016522,0.120000
4,95,0.0010,30,160,100,190,0.020067,0.106667
5,101,0.0030,30,200,80,40,0.254441,0.100000
6,102,0.0010,32,110,160,70,0.013360,0.146667
7,112,0.0030,30,200,80,40,0.254441,0.100000
8,114,0.0002,120,90,40,10,0.018673,0.113333
9,148,0.0002,120,90,10,10,0.013640,0.133333


In [52]:
from optuna.visualization import plot_optimization_history

for i in range(len(names_objs)):
    plot_optimization_history(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() 

In [53]:
from optuna.trial import TrialState
completed = [t for t in study.trials if t.state == TrialState.COMPLETE]
print("Number of trials:", len(study.trials))
print("Number of completed trials:", len(completed))

Number of trials: 150
Number of completed trials: 145


In [54]:
from optuna.visualization import plot_pareto_front
plot_pareto_front(
    study,
    targets=lambda t: [t.values[0], t.values[1]],
    target_names=["Survival curves dist", "Identifiability score"],
    include_dominated_trials=True
)

In [56]:
selected_trial_id = 95
best_trial = study.trials[selected_trial_id]

print("Values (objectifs):", best_trial.values)
print("Params:", best_trial.params)
print("State:", best_trial.state)
print("Start:", best_trial.datetime_start)
print("End:", best_trial.datetime_complete)
print("Duration:", best_trial.duration)

Values (objectifs): [0.020067247068494207, 0.10666666666666667]
Params: {'lr': 0.001, 'batch_size': 30, 'z_dim': 160, 'y_dim': 100, 's_dim': 190}
State: 1
Start: 2026-06-17 17:05:02.659622
End: 2026-06-17 17:09:07.415047
Duration: 0:04:04.755425


### Save best selected trial

In [57]:
import json
parent_path = "/Users/pchassat/Documents/survgen-clinical-trials"
best_params_file = parent_path + "/dataset/" + dataset + "/optuna_results/best_params_{}_ntrials{}_{}_{}.json".format(name_config, n_trials, optuna_version_name, generator_name)
# best_params_file = parent_path + "/dataset/" + dataset + "/optuna_results/best_params_traincontrol_{}_ntrials{}_{}_{}.json".format(dataset, n_trials, optuna_version_name, generator_name)
with open(best_params_file, "w") as f:
    json.dump(best_trial.params, f)